# Document-Based Stores (MongoDB)

### Task 1: Create a simple MongoDB out of this relational model

This is  a toy DB about movies and actors who played roles in these movies. This DB is consisted of  

- A "Person" table who has a unique id, and a name fields.

- Another "Movie" table that has a unique id, a title, a country where it was made, and a year when it was released.

- There is (m-n) or "many-many" relationship between these two tables (i.e basically, many actors can act in many movies, and the movie include many actors)
- Therefore, we use the "Roles" table in which we can deduct which person has acted in which movie, and what role(s) they played.

<img src="RDBSchema.png" alt="3" border="0">

#### Connect to the MongoDB server, and create a mongoDB with the name 'moviedb'

In [1]:
from pymongo import MongoClient
from random import randint
from pprint import pprint

import warnings
warnings.filterwarnings('ignore')

In [2]:
mongo = "mongo"

# we use the MongoClient to communicate with the running database instance.
myclient = MongoClient(
                    "mongodb://"+mongo+":27017/",  
                    username='admin',
                    password='admin') #Mongo URI format

mydb = myclient["moviedb"]

#### Create Person/Actor collection

In [3]:
person_coll = myclient["person"]

#### Insert the data into the Person Table

In [8]:
personList = [
  { "id": 1, "name": "Charlie Sheen" },
  { "id": 2, "name": "Michael Douglas"},
  { "id": 3, "name": "Martin Sheen"},
  { "id": 4, "name": "Morgan Freeman"}
]

persons = mydb.person_coll.insert_many(personList)

#### Creating rest of Collections ("Movies", "Roles")

In [5]:
restcols = ["Movies","Roles"]

for col in restcols:
    myclient[col]

#### Inserting data into the movie Collection

In [9]:
moviescoll = myclient["Movies"]

movieList = [
  { "id": 1, "title": "Wall Street", "country":"USA","year":1987},
  { "id": 2, "title": "The American President", "country":"USA","year":1995},
  { "id": 3, "title": "The Shawshank Redemption", "country":"USA","year":1994},
]

movies = mydb.moviescoll.insert_many(movieList)

#### Inserting data into the roles Collection

In [10]:
rolesCol = myclient["Roles"]

roleList = [
  { "personId": 1, "movieId": 1, "role":["Bud Fox"]},
  { "personId": 2, "movieId": 1, "role":["Carl Fox"]},
  { "personId": 3, "movieId": 1, "role":["Gordon Gekko"]},
  { "personId": 2, "movieId": 2, "role":["A.J. MacInerney"]},
  { "personId": 3, "movieId": 2, "role":["President Andrew Shepherd"]},
  { "personId": 4, "movieId": 3, "role":["Ellis Boyd 'Red' Redding"]}
]

roles = mydb.rolesCol.insert_many(roleList)

In [52]:
mydb.rolesCol.insert_one({ "personId": 4, "movieId": 1, "role":["Sissoko Ndiaye"]})

InsertOneResult(ObjectId('68e7e3949ad164d5fa13c1bf'), acknowledged=True)

### <font color ='green'>Just for your info</font>:

#### Another Way of Modeling this M-N model in Mongo would be using the Forien Keys 


* Movies


```[

{
	"_id": 1,
	"title":"Wall Street",
	"country":"USA",
	"year":1987,
	"persons":[1,2]
},

{
	"_id": 2,
	"title":"The American President",
	"country":"USA",
	"year":1995,
	"persons":[2]
}]
```
* Actors

```
[{
    "_id": 1,
    "name": "Charlie Sheen",
    "movies":[
    {"role": "Bud Fox", "movie_id":1}
    ]
},

{
    "_id": 2,
    "name": "Micheal Douglas",
    "movies":[
    {"role": "Gordon Geko", "movie_id":1},
    {"role": "President Andrew Shepherd", "movie_id":2}
    ]
}

] ```


#### Get all actors in your Mongo DB

In [20]:
allActors = mydb.person_coll.find({})
for a in allActors:
    print(a)

{'_id': ObjectId('68e67579e84f5c7336a7b18f'), 'id': 1, 'name': 'Charlie Sheen'}
{'_id': ObjectId('68e67579e84f5c7336a7b190'), 'id': 2, 'name': 'Michael Douglas'}
{'_id': ObjectId('68e67579e84f5c7336a7b191'), 'id': 3, 'name': 'Martin Sheen'}
{'_id': ObjectId('68e67579e84f5c7336a7b192'), 'id': 4, 'name': 'Morgan Freeman'}


#### Get actors with names start with 'C' letter

In [24]:
c_actors = mydb.person_coll.find({
    "name": {
        "$regex": "^C"
    }
})

for a in c_actors:
    print(a)

{'_id': ObjectId('68e67579e84f5c7336a7b18f'), 'id': 1, 'name': 'Charlie Sheen'}


#### Get all Movies sorted from recent to old! (get only the title and year fields)

In [27]:
allMovies = mydb.moviescoll.find({}, {"title": 1, "year": 1}).sort({"year": 1})
for mov in allMovies:
    print(mov)

{'_id': ObjectId('68e67632e84f5c7336a7b193'), 'title': 'Wall Street', 'year': 1987}
{'_id': ObjectId('68e67632e84f5c7336a7b195'), 'title': 'The Shawshank Redemption', 'year': 1994}
{'_id': ObjectId('68e67632e84f5c7336a7b194'), 'title': 'The American President', 'year': 1995}


#### Get all Movies released in the 90s (after year (1990) and before 2000) ordered from old to recent.

In [37]:
movies_90_2000 = mydb.moviescoll.find({
    "year": {
        "$gt": 1990,
        "$lt": 2000
    }
}).sort({"year": 1})

for m in movies_90_2000:
    print(m)

{'_id': ObjectId('68e67632e84f5c7336a7b195'), 'id': 3, 'title': 'The Shawshank Redemption', 'country': 'USA', 'year': 1994}
{'_id': ObjectId('68e67632e84f5c7336a7b194'), 'id': 2, 'title': 'The American President', 'country': 'USA', 'year': 1995}


#### Get Movies and Actors from your "movies" DB
* Hint : use the <code>'$lookup'</code> operator.
* The Result should be something like the following:
<code>
Charlie Sheen : Wall Street
Michael Douglas : Wall Street
Martin Sheen : Wall Street
Michael Douglas : The American President
Martin Sheen : The American President
Morgan Freeman : The Shawshank Redemption
</code>

In [39]:
allRoles = mydb.rolesCol.find({})
all = mydb.rolesCol.aggregate([
    {
        "$lookup": {
            "from": "person_coll",
            "localField": "personId",
            "foreignField": "id",
            "as": "persons"
        }
    },
    {
        "$unwind": "$persons"
    },
    {
        "$lookup": {
            "from": "moviescoll",
            "localField": "movieId",
            "foreignField": "id",
            "as": "movies"
        }
    },
    {
        "$unwind": "$movies"
    },
    {
        "$project": {
            "_id": 0,
            "name": "$persons.name",
            "title": "$movies.title"
        }
    }
])

for r in all:
    print(r)
    # print(r['persons']['name']+' : '+r['movies']['title'])

{'name': 'Charlie Sheen', 'title': 'Wall Street'}
{'name': 'Michael Douglas', 'title': 'Wall Street'}
{'name': 'Martin Sheen', 'title': 'Wall Street'}
{'name': 'Michael Douglas', 'title': 'The American President'}
{'name': 'Martin Sheen', 'title': 'The American President'}
{'name': 'Morgan Freeman', 'title': 'The Shawshank Redemption'}


#### For each Actor, get count of "Movies" he acted in.

In [50]:
res = mydb.rolesCol.aggregate([
    { "$group": { "_id": "$personId", "Movie Count": { "$count": {}}} },
    { 
        "$project": { 
            "_id": 0,
            "Actor ID": "$_id",
            "Movies": "$Movie Count"
        }
    }
])

for row in res:
    print(row)

{'Actor ID': 2, 'Movies': 2}
{'Actor ID': 3, 'Movies': 2}
{'Actor ID': 1, 'Movies': 1}
{'Actor ID': 4, 'Movies': 1}


#### In your DB, list the movies that every Actor played

In [98]:
totalActors = mydb.person_coll.count_documents({})
filmsWithAllActors = mydb.rolesCol.aggregate([
    { "$group": { "_id": "$movieId", "nbActors": { "$count": {}}} },
    { "$match": { "nbActors": { "$gte": totalActors }} },
    {
    "$lookup": {
            "from": "moviescoll",      # La collection à joindre (cible)
            "localField": "_id",       # Le champ de jointure dans le document courant (c'est le movieId)
            "foreignField": "id",     # Le champ de jointure dans la collection cible (c'est l'ID unique du film)
            "as": "movie_details"      # Le nom du tableau de sortie
        }
    },
    {"$unwind": "$movie_details" },
    { "$project": { "_id": 0, "Name": "$movie_details.title"} }
])
for r in filmsWithAllActors:
    print(r["Name"])

Wall Street


#### Get the Persons/Actors who acted in "Wall Street" movie
- Hint use `$lookup` , `$match` operators in the aggregation piepeline

In [105]:
res = mydb.rolesCol.aggregate([
    { "$lookup": {
            "from": "moviescoll",
            "localField": "movieId",
            "foreignField": "id",
            "as": "movie_details"
        }
    },
    {
        "$unwind": "$movie_details"
    },
    { "$match": { "movie_details.title": "Wall Street"}},
    { "$project": { "_id": 0, "personId": 1 }}
])
for r in res:
    print(r)

{'personId': 1}
{'personId': 2}
{'personId': 3}
{'personId': 4}


#### Get the Movies in which "Michael Douglas" has played a role in

In [115]:
res = mydb.rolesCol.aggregate([
    { "$lookup": {
        "from": "person_coll",
        "localField": "personId",
        "foreignField": "id",
        "as": "person_details"
    }},
    { "$unwind": "$person_details" },
    { "$match": {"person_details.name": "Michael Douglas"}},
    { "$project": { "_id": 0, "movieId": 1}}
])

for r in res:
    print(r)

{'movieId': 1}
{'movieId': 2}


#### Get count of "Movies" in your DB

In [116]:
mydb.moviescoll.count_documents({})

3

#### update the year of the 'Wall Street' movie was released in to be 2000(which is not true BTW :)
- Show that movie before and After updating it

In [124]:
wall_street_movie = mydb.moviescoll.find_one({"title": "Wall Street"})
print(wall_street_movie)

mydb.moviescoll.find_one_and_update({"title": "Wall Street"}, { "$set": {"year": 2000}})

wall_street_movie = mydb.moviescoll.find_one({"title": "Wall Street"})
print(wall_street_movie)

{'_id': ObjectId('68e7d9eb9ad164d5fa13c1b6'), 'id': 1, 'title': 'Wall Street', 'country': 'USA', 'year': 1987}
{'_id': ObjectId('68e7d9eb9ad164d5fa13c1b6'), 'id': 1, 'title': 'Wall Street', 'country': 'USA', 'year': 2000}


####  Delete all the persons with names start with 'M' letter.

In [129]:
for a in mydb.person_coll.find({}):
    print(a)

mydb.person_coll.delete_many({"name": { "$regex": "^M"}})

for a in mydb.person_coll.find({}):
    print(a)

{'_id': ObjectId('68e7d9e99ad164d5fa13c1b2'), 'id': 1, 'name': 'Charlie Sheen'}
{'_id': ObjectId('68e7d9e99ad164d5fa13c1b3'), 'id': 2, 'name': 'Michael Douglas'}
{'_id': ObjectId('68e7d9e99ad164d5fa13c1b4'), 'id': 3, 'name': 'Martin Sheen'}
{'_id': ObjectId('68e7d9e99ad164d5fa13c1b5'), 'id': 4, 'name': 'Morgan Freeman'}
{'_id': ObjectId('68e7d9e99ad164d5fa13c1b2'), 'id': 1, 'name': 'Charlie Sheen'}


### Task 2: Extend your Mongo-"MovieDB" 

Imagine now that we are going to extend our DB with new movies, actors, even with new directors.

- We add <b>**"The matrix"**</b> movie which was released in <b> USA, (1999)</b>, and has a new property/field "Tagline" <b>("Welcome to the Real World")</b>.
 
- We will also add 4 new actors (Person):
    - **"Keanu Reeves"** who was born in (1964). <font color='green'>Note:</font> "born" property is also new.
    - **"Carrie-Anne Moss"** who was born in (1967).
    - **"Laurence Fishburne"** who was born in (1960).
    - **"Hugo Weaving"** who was born in (1960).
    
- Moreover, we add 2 directors (Person) :
    - **"Lilly Wachowski"**, born in (1967)
    - **"Lana Wachowski"**, born in(1965)
- For these directors specify one more label/field as ("Director"). (You can add this while inserting the director documents)
    
 
- We will also create a new <b>collection "Directed" </b> that is directed from the later 2 directors to "the Matrix" movie.

#### Add the Movie "The Matrix" with the provided data to the Movies collection

In [137]:
mydb.moviescoll.insert_one({
    "id": 4,
    "title": "The Matrix",
    "country": "USA",
    "year": 1999
})

InsertOneResult(ObjectId('68e7f6b19ad164d5fa13c1c9'), acknowledged=True)

#### Insert the new 4 actors to the person collection

In [132]:
#Notice, How is easy to add a new feild compared to the RDB
newActorList = [
  { "id": 5, "name": "Keanu Reeves", "born":1964 },
  { "id": 6, "name": "Carrie-Anne Moss", "born":1967},
  { "id": 7, "name": "Laurence Fishburne", "born":1960},
  { "id": 8, "name": "Hugo Weaving", "born":1960}
]

mydb.person_coll.insert_many(newActorList)

InsertManyResult([ObjectId('68e7f4cc9ad164d5fa13c1c1'), ObjectId('68e7f4cc9ad164d5fa13c1c2'), ObjectId('68e7f4cc9ad164d5fa13c1c3'), ObjectId('68e7f4cc9ad164d5fa13c1c4')], acknowledged=True)

#### Insert the new 2 directors to the person collection

In [136]:
newPersonsList = [
  {  "id": 9, "name": "Lilly Wachowski", "born":1967, "director": True},
  {  "id": 10, "name": "Lana Wachowski", "born":1960, "director": True},
]

mydb.person_coll.insert_many(newPersonsList)

InsertManyResult([ObjectId('68e7f68b9ad164d5fa13c1c7'), ObjectId('68e7f68b9ad164d5fa13c1c8')], acknowledged=True)

#### Create the "Directed" collection, and insert the data into it 

In [182]:
directed_coll = mydb["Directed"]

newDirectorList = [
  {  "movieId": 4, "personId": 9},
  {  "movieId": 4, "personId": 10},
]

In [184]:
directed_coll.insert_many(newDirectorList)

InsertManyResult([ObjectId('68e7fbdd9ad164d5fa13c1cc'), ObjectId('68e7fbdd9ad164d5fa13c1cd')], acknowledged=True)

#### Get only the directors from the person collection (i.e. persons marked with the label "Director")

In [143]:
directors = mydb.person_coll.find({"director": True})
for director in directors:
    print(director)

{'_id': ObjectId('68e7f68b9ad164d5fa13c1c7'), 'id': 9, 'name': 'Lilly Wachowski', 'born': 1967, 'director': True}
{'_id': ObjectId('68e7f68b9ad164d5fa13c1c8'), 'id': 10, 'name': 'Lana Wachowski', 'born': 1960, 'director': True}


#### Perform a query that get persons (names, and born year) who Directed "The Matrix" movie.

In [196]:
res = mydb.moviescoll.aggregate([
    { 
        "$match": { "title": "The Matrix"}
    },
    {
        "$lookup": {
            "from": "Directed",
            "localField": "id",
            "foreignField": "movieId",
            "as": "directed_details"
        }
    },
    {
        "$unwind": "$directed_details"
    },
    {
        "$lookup": {
            "from": "person_coll",
            "localField": "directed_details.personId",
            "foreignField": "id",
            "as": "persons_details"
        }
    },
    {
        "$unwind": "$persons_details"
    },
    {
        "$project": {"_id": 0, "Name": "$persons_details.name", "Born": "$persons_details.born"}
    }
])

for r in res:
    print(r)

{'Name': 'Lilly Wachowski', 'Born': 1967}
{'Name': 'Lana Wachowski', 'Born': 1960}


 ## How long did it take you to solve the homework?
 
Please answer as precisely as you can. It does not affect your points or grade in any way. It is okey, if it took 0.5 hours or 24 hours. The collected information will be used to improve future homeworks.

<font color="red"><b>Answer:</b></font> 
Note: (Almost) No generative AI has been used + No prior background with MongoDB.
Time taken: 3h

**<center> <font color='red'>THANK YOU FOR YOUR EFFORT!</font></center>**